# Search YOUR Existing Embeddings Table (text-embedding-004)

Your documents are **already embedded and stored** — this notebook only does the query side:

1. **Inspect** your table so you know its exact structure
2. **Embed** the user's question with `text-embedding-004` (the SAME model that made your stored vectors — this is critical)
3. **Search** with pgvector's cosine-distance operator `<=>` to get the most relevant rows
4. *(Optional)* Feed those rows to Gemini for a natural-language answer

**Before running:** `gcloud auth application-default login` in a terminal, and fill in the CONFIG cell.

## Step 0 — Install dependencies (run once)

In [ ]:
%pip install -q "cloud-sql-python-connector[pg8000]" sqlalchemy pgvector google-cloud-aiplatform google-genai

## Step 1 — Configuration

Fill in your Cloud SQL details. For the three `EXISTING TABLE` values, make your best guess now — **Step 3 will inspect the database and show you the real names**, then come back and correct these if needed.

In [ ]:
# ---- GCP / Cloud SQL ----
PROJECT_ID = "your-gcp-project-id"
REGION = "asia-south1"
INSTANCE_CONNECTION_NAME = "your-project:your-region:your-instance"
DB_USER = "postgres"
DB_PASS = "your-db-password"
DB_NAME = "postgres"

# ---- YOUR EXISTING TABLE (Step 3 will help you verify these) ----
TABLE_NAME = "your_table_name"        # the table that holds the embeddings
CONTENT_COLUMN = "content"            # column with the document text
EMBEDDING_COLUMN = "embedding"        # column with the vectors

# ---- Models ----
EMBEDDING_MODEL = "text-embedding-004"   # MUST match the model that created your stored embeddings
CHAT_MODEL = "gemini-2.5-flash"          # only used in the optional last step

## Step 2 — Connect to Cloud SQL

In [ ]:
import sqlalchemy
from google.cloud.sql.connector import Connector

connector = Connector()

def _get_conn():
    return connector.connect(
        INSTANCE_CONNECTION_NAME,
        "pg8000",
        user=DB_USER,
        password=DB_PASS,
        db=DB_NAME,
    )

engine = sqlalchemy.create_engine("postgresql+pg8000://", creator=_get_conn)

# quick connectivity test
with engine.connect() as db:
    print("Connected! Postgres says:", db.execute(sqlalchemy.text("SELECT version()")).scalar()[:40], "...")

## Step 3 — Inspect your existing table

Run these cells to discover what's actually in your database. First: list all tables that contain a `vector` column — one of them is yours.

In [ ]:
with engine.connect() as db:
    rows = db.execute(sqlalchemy.text("""
        SELECT table_name, column_name, udt_name
        FROM information_schema.columns
        WHERE table_schema = 'public' AND udt_name = 'vector'
    """)).fetchall()

if rows:
    print("Tables with a vector (embedding) column:")
    for r in rows:
        print(f"  table: {r.table_name:30s} embedding column: {r.column_name}")
    print("\n-> Set TABLE_NAME and EMBEDDING_COLUMN in Step 1 to match, then re-run that cell.")
else:
    print("No vector columns found — is pgvector enabled and is this the right database (DB_NAME)?")

Now look at ALL columns of your table (so you can spot the text/content column), check the vector dimension, and preview a row. text-embedding-004 produces **768** dimensions — the printed dimension should say 768.

In [ ]:
with engine.connect() as db:
    # all columns of the table
    cols = db.execute(sqlalchemy.text("""
        SELECT column_name, udt_name
        FROM information_schema.columns
        WHERE table_schema = 'public' AND table_name = :t
        ORDER BY ordinal_position
    """), {"t": TABLE_NAME}).fetchall()
    print(f"Columns of '{TABLE_NAME}':")
    for c in cols:
        print(f"  {c.column_name:25s} {c.udt_name}")

    # row count + vector dimension
    n = db.execute(sqlalchemy.text(f'SELECT COUNT(*) FROM {TABLE_NAME}')).scalar()
    dim = db.execute(sqlalchemy.text(
        f'SELECT vector_dims({EMBEDDING_COLUMN}) FROM {TABLE_NAME} LIMIT 1'
    )).scalar()
    print(f"\nRows: {n}")
    print(f"Vector dimension: {dim}  (should be 768 for text-embedding-004)")

    # preview one row's text
    sample = db.execute(sqlalchemy.text(
        f'SELECT {CONTENT_COLUMN} FROM {TABLE_NAME} LIMIT 1'
    )).scalar()
    print(f"\nSample {CONTENT_COLUMN}: {str(sample)[:200]}")

## Step 4 — Embed the question with text-embedding-004

**The golden rule:** the question must be embedded by the *same model* that embedded your documents. Different models produce vectors in different coordinate systems — mixing them makes distances meaningless.

`task_type="RETRIEVAL_QUERY"` tells the model this is a search question (your documents were most likely stored with `RETRIEVAL_DOCUMENT`).

In [ ]:
from google import genai
from google.genai.types import EmbedContentConfig

ai = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

def embed_question(question: str) -> list[float]:
    resp = ai.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=[question],
        config=EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    )
    return resp.embeddings[0].values

# test it
v = embed_question("test question")
print(f"Got a {len(v)}-dimensional vector (must match Step 3's dimension). First 5: {v[:5]}")

## Step 5 — Search your table

This is the retrieval step. `<=>` is pgvector's **cosine distance**: 0 = identical meaning, ~1 = unrelated. Postgres sorts every row by distance to your question's vector and returns the closest `top_k`.

In [ ]:
def search(question: str, top_k: int = 5) -> list[dict]:
    qvec = embed_question(question)

    with engine.connect() as db:
        rows = db.execute(
            sqlalchemy.text(f"""
                SELECT {CONTENT_COLUMN} AS content,
                       {EMBEDDING_COLUMN} <=> :qvec AS distance
                FROM {TABLE_NAME}
                ORDER BY distance
                LIMIT :top_k
            """),
            {"qvec": str(qvec), "top_k": top_k},
        ).fetchall()

    return [{"content": r.content, "distance": float(r.distance)} for r in rows]

Try it with a real question about YOUR documents. Lower distance = more relevant. If the results look wrong, see the troubleshooting notes at the bottom.

In [ ]:
results = search("PUT A QUESTION ABOUT YOUR DOCUMENTS HERE", top_k=5)

for i, r in enumerate(results, 1):
    print(f"--- Result {i}  (distance: {r['distance']:.4f}) ---")
    print(r["content"][:300])
    print()

## Step 6 (optional) — Let Gemini write the answer

Instead of returning raw chunks, paste them into a prompt and let Gemini compose a natural answer grounded in your data.

In [ ]:
def ask(question: str, top_k: int = 5) -> str:
    hits = search(question, top_k)
    context = "\n\n---\n\n".join(h["content"] for h in hits)

    prompt = f"""Answer the question using ONLY the context below.
If the context doesn't contain the answer, say you don't know.

CONTEXT:
{context}

QUESTION: {question}"""

    resp = ai.models.generate_content(model=CHAT_MODEL, contents=prompt)
    return resp.text

print(ask("PUT A QUESTION ABOUT YOUR DOCUMENTS HERE"))

## Troubleshooting

- **Wrong/irrelevant results:** almost always a model mismatch — confirm the stored vectors were really made by `text-embedding-004`. Also try `task_type=None` in `embed_question` (if the documents were embedded without a task type, matching without one can work better).
- **`vector_dims` error or dimension ≠ 768:** the embedding column may not be a pgvector `vector` type, or the stored model wasn't text-embedding-004. Tell me what Step 3 printed and I'll adapt the code.
- **Slow searches on a large table:** add an index once:
  `CREATE INDEX ON your_table USING hnsw (embedding vector_cosine_ops);`
- **Model deprecation note:** text-embedding-004 is an older model — fine to keep using, but if Google retires it you'd need to re-embed all documents with a newer model (e.g. gemini-embedding-001), because old and new vectors can't be mixed.